# 02 Nighttime Scenery 50MP (Phase 3)

**Synthetic RAW 50MP Lab.** This notebook builds a 50 MP night master from an approximately 4K AI-generated
night image (PRD sections 4.2, 8.3, 12 Layer 5B, 19 and 30 Phase 3):

- **Scene analysis:** PRD 8.3 luminance regions (near-black, low-luminance textured, midtone, bright practical
  lights, potentially clipped highlights, smooth gradients, specular reflections, sky and atmosphere), shown
  as a mask and correctable by hand.
- **Reconstruction:** the bicubic and Lanczos baselines and the conservative and balanced SR slots, run by the
  Phase 2 pretrained-SR backend (SwinIR by default, inference only). The configured mode (balanced by default)
  is the night input. SR detail is **inferred**: synthesised from learned priors, not recovered, and labelled
  as such everywhere.
- **Layer 5B night processing:** shadow handling, highlight protection, night colour, conservative chroma-noise
  control, optional atmospheric contrast, grain and bloom, with the PRD 12.6 failure rule.
- **Outputs:** 16-bit TIFF and half-float OpenEXR masters under `outputs/night/`, the PRD 19.3 comparisons and
  the PRD 23.5 night metrics before and after processing, in the run report.

All processing code lives in the shared `synthetic_raw_lab` package (`src/night_processing.py` and
`src/night_pipeline.py`); this notebook only sets parameters, calls the package and displays results.

## 1. Introduction and scientific limitations

- The source is a rendered, **display-referred** night image. Converting it to linear light gives linear-light
  display-referred values, **not scene-referred radiance**.
- **Clipped highlights stay clipped.** The highlight shoulder rolls clipped lights off smoothly so they look less
  harsh; it never recovers radiance the source did not record, and the clipping metrics say so.
- **Shadow lift redistributes existing tones.** Deep shadows are lifted gently with black kept at black; texture
  is not invented in near-black regions (low invented-texture tolerance).
- Pretrained super-resolution **synthesises** detail above the source sampling limit from priors learned on
  other photographs. That detail is inferred, not recovered, and the SR slots are reported separately from the
  baselines.
- Debanding smooths steps smaller than two 8-bit codes; dither, grain and bloom are **simulated** and labelled.
- Every output and step carries a provenance label: **preserved**, **estimated**, **synthesized** or
  **simulated**. Where processing creates halos, banding, black crush or excess texture, the failure rule falls
  back to a more conservative local result and records where it happened.
- Outputs are **synthetic camera-grade night masters**, never camera RAW files.

## 2. Runtime setup

Set the run parameters. In Google Colab these are form fields.

- **Repository access, code version and keeping masters:** as in notebook 00 (public clone, optional
  `GITHUB_TOKEN` Colab secret or `PROJECT_DIR`; `REPO_REF`; `MOUNT_DRIVE` with `OUTPUT_ROOT` on Drive). A 50 MP night run writes several
  hundred MB.
- **Input:** `synthetic` generates the deterministic synthetic night scene (banding-prone sky gradient, clipped
  street lights and windows, coloured neon, wet-road reflections, noisy shadows) at `SYNTHETIC_SIZE`.
- **Presets (PRD 21.6):** `quick` makes a reduced output (`quick_target_megapixels` in the profile, 2 MP by
  default); `standard` the full 50 MP target; `research` also benchmarks memory.
- **Super-resolution:** `ENABLE_SUPER_RESOLUTION` runs the conservative and balanced SR slots (PRD 19.3) with
  the `SR_BACKEND` networks, and Layer 5B processes the `SR_MODE` result. Off, the night input is the Lanczos
  baseline and no SR is claimed. Weights (about 135 MB for SwinIR) download once into `WEIGHTS_DIR` (default
  `<project>/weights`; a Drive folder keeps them across sessions) and are checksum-verified. Use a GPU runtime:
  50 MP SR takes minutes on a GPU and much longer on the CPU. Model choice and licences: `docs/models.md`.
- **PRD 19.2 night controls:** leave a field empty to keep the value from `CONFIG_FILE`. Section 7 lists every
  parameter with its accepted values, effect and warning. Levels such as `low` can be replaced by a number.
- **Advanced (PRD 22.2):** `ADVANCED` is a dictionary of any further profile keys, flat (for example
  `{"crops": [[x, y, w, h]]}`) or by section (for example `{"failure_rule": {"block_size": 64}}`).
  `REGION_OVERRIDES` corrects the PRD 8.3 region masks by hand: a list of
  `{"region": name, "boxes": [[x, y, width, height]], "note": text}` in source pixels.

In [ ]:
# @title Run parameters
REPO_URL = "https://github.com/RDC28/experiment_50MP.git"  # @param {type:"string"}
REPO_REF = "main"  # @param {type:"string"}
PROJECT_DIR = ""  # @param {type:"string"}
MOUNT_DRIVE = False  # @param {type:"boolean"}
DOWNLOAD_FINAL = True  # @param {type:"boolean"}
FINAL_DRIVE_DIR = "/content/drive/MyDrive/experiment_50MP/final"  # @param {type:"string"}
OUTPUT_ROOT = ""  # @param {type:"string"}
REPORTS_ROOT = ""  # @param {type:"string"}
INPUTS_ROOT = ""  # @param {type:"string"}
CONFIG_FILE = "configs/night_default.yaml"  # @param {type:"string"}
INPUT_MODE = "upload"  # @param ["upload", "path", "synthetic"]
INPUT_PATH = ""  # @param {type:"string"}
SYNTHETIC_SIZE = "3840x2160"  # @param {type:"string"}
RUNTIME_PRESET = "standard"  # @param ["quick", "standard", "research"]
TARGET_MEGAPIXELS = 50.0  # @param {type:"number"}
CROP_SIZE = 256  # @param {type:"integer"}
ENABLE_SUPER_RESOLUTION = True  # @param {type:"boolean"}
SR_BACKEND = "swinir"  # @param ["swinir", "realesrgan"]
WEIGHTS_DIR = ""  # @param {type:"string"}

# @markdown PRD 19.2 night controls (empty keeps the CONFIG_FILE value)
SR_MODE = ""  # @param ["", "conservative", "balanced"]
SHADOW_LIFT = ""  # @param ["", "none", "low", "medium", "high"] {allow-input: true}
SHADOW_COLOR_PRESERVATION = ""  # @param {type:"string"}
BLACK_POINT = ""  # @param {type:"string"}
HIGHLIGHT_SHOULDER = ""  # @param ["", "none", "low", "moderate", "strong"] {allow-input: true}
BLOOM_AMOUNT = ""  # @param {type:"string"}
HALO_SUPPRESSION = ""  # @param ["", "true", "false"]
DENOISE_STRENGTH = ""  # @param ["", "off", "conservative", "moderate", "strong"] {allow-input: true}
BANDING_REDUCTION = ""  # @param ["", "true", "false"]
ARCHITECTURAL_DETAIL = ""  # @param ["", "none", "minimal", "low", "moderate", "high"] {allow-input: true}
ATMOSPHERIC_SOFTNESS = ""  # @param {type:"string"}
COLOR_TEMPERATURE_K = ""  # @param {type:"string"}
GRAIN_AMOUNT = ""  # @param {type:"string"}
SHARPENING_RADIUS = ""  # @param {type:"string"}
SHARPENING_STRENGTH = ""  # @param {type:"string"}

# @markdown Advanced parameter dictionaries
ADVANCED = {}  # @param {type:"raw"}
REGION_OVERRIDES = []  # @param {type:"raw"}

## 3. Dependency installation

Finds the repository (`PROJECT_DIR`, the checkout this notebook lives in, or the Colab clone of `REPO_REF`)
and installs the `synthetic_raw_lab` package from it (plus the `sr` extra, PyTorch and spandrel, when
super-resolution is enabled), exactly like notebook 00. The exact package versions used are recorded in every
run manifest.

In [ ]:
import base64
import importlib
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

NOTEBOOK_VERSION = "02_Night_Scenery_50MP/1.2"
COLAB_CLONE = Path("/content/experiment_50MP")
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if MOUNT_DRIVE and IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")


def find_project() -> Path:
    if PROJECT_DIR:
        return Path(PROJECT_DIR).expanduser().resolve()
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "pipeline.py").exists():
            return candidate.resolve()
    return COLAB_CLONE


def git(*args: str) -> None:
    env = dict(os.environ)
    token = ""
    if IN_COLAB:
        try:
            from google.colab import userdata

            token = userdata.get("GITHUB_TOKEN") or ""
        except Exception:
            token = ""
    if token:
        credential = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env.update(
            GIT_CONFIG_COUNT="1",
            GIT_CONFIG_KEY_0="http.extraHeader",
            GIT_CONFIG_VALUE_0=f"Authorization: Basic {credential}",
        )
    result = subprocess.run(["git", *args], env=env, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(
            f"git failed (check REPO_URL and REPO_REF; a private fork needs the GITHUB_TOKEN secret): {result.stderr}"
        )


PROJECT = find_project()
if not (PROJECT / "pyproject.toml").exists():
    git("clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(PROJECT))
elif PROJECT == COLAB_CLONE:
    git("-C", str(PROJECT), "fetch", "--depth", "1", REPO_URL, REPO_REF)
    git("-C", str(PROJECT), "checkout", "--detach", "FETCH_HEAD")
spec = importlib.util.find_spec("synthetic_raw_lab")
missing_sr = ENABLE_SUPER_RESOLUTION and importlib.util.find_spec("spandrel") is None
if spec is None or Path(spec.origin).resolve().parent != PROJECT / "src" or missing_sr:
    target = f"{PROJECT}[sr]" if ENABLE_SUPER_RESOLUTION else str(PROJECT)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", target], check=True)
    importlib.invalidate_caches()
for name in [m for m in sys.modules if m.partition(".")[0] == "synthetic_raw_lab"]:
    del sys.modules[name]

import synthetic_raw_lab  # noqa: E402
from synthetic_raw_lab import environment  # noqa: E402

print(f"project: {PROJECT} (Git: {environment.git_revision(PROJECT)})")
print(f"synthetic_raw_lab {synthetic_raw_lab.__version__} from {Path(synthetic_raw_lab.__file__).parent}")

## 4. GPU and environment inspection

Loads the night profile (`CONFIG_FILE`, then the form fields and `ADVANCED`; empty path fields keep the
file's paths) so every later cell uses the same settings as the pipeline. Then reports GPU, CUDA driver,
PyTorch (if installed), the device super-resolution will use, RAM and free disk where the masters will be
written (PRD 21.2). Super-resolution uses CUDA when present, then Apple MPS, then the CPU, with one network
loaded at a time. The baselines and Layer 5B run on the CPU, in strips.

In [ ]:
from IPython.display import Image, Markdown, display

from synthetic_raw_lab import config as lab_config
from synthetic_raw_lab import environment, ingestion, night_pipeline, pipeline, resolution, sr_models
from synthetic_raw_lab import night_processing as night


def table(header, rows):
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(v).replace("|", "\\|") for v in row) + " |" for row in rows]
    return Markdown("\n".join(lines))


def control(text):
    """A PRD 19.2 form value: empty keeps the profile value, true/false are booleans, numbers are numbers."""
    text = str(text).strip()
    if text.lower() in ("true", "false"):
        return text.lower() == "true"
    try:
        return float(text)
    except ValueError:
        return text


CONTROLS = {
    ("super_resolution", "mode"): SR_MODE,
    ("shadows", "lift_strength"): SHADOW_LIFT,
    ("shadows", "color_preservation"): SHADOW_COLOR_PRESERVATION,
    ("shadows", "black_point"): BLACK_POINT,
    ("highlights", "rolloff_strength"): HIGHLIGHT_SHOULDER,
    ("optics", "bloom_amount"): BLOOM_AMOUNT,
    ("highlights", "halo_suppression"): HALO_SUPPRESSION,
    ("noise", "denoise_source"): DENOISE_STRENGTH,
    ("shadows", "banding_reduction"): BANDING_REDUCTION,
    ("texture", "architecture_detail_strength"): ARCHITECTURAL_DETAIL,
    ("texture", "atmospheric_softness"): ATMOSPHERIC_SOFTNESS,
    ("color", "color_temperature_k"): COLOR_TEMPERATURE_K,
    ("noise", "grain_amount"): GRAIN_AMOUNT,
    ("texture", "sharpening_radius"): SHARPENING_RADIUS,
    ("texture", "sharpening_strength"): SHARPENING_STRENGTH,
}
overrides = {
    "runtime_preset": RUNTIME_PRESET,
    "target_megapixels": float(TARGET_MEGAPIXELS),
    "crop_size": CROP_SIZE,
    "super_resolution": {"enabled": ENABLE_SUPER_RESOLUTION, "backend": SR_BACKEND},
}
if WEIGHTS_DIR:
    overrides["super_resolution"]["weights_dir"] = WEIGHTS_DIR
paths = {"output_root": OUTPUT_ROOT, "reports_root": REPORTS_ROOT, "inputs_root": INPUTS_ROOT}
overrides.update({name: value for name, value in paths.items() if value})
for (section, key), text in CONTROLS.items():
    if str(text).strip():
        overrides.setdefault(section, {})[key] = control(text)
for key, value in dict(ADVANCED).items():
    if isinstance(value, dict) and isinstance(overrides.get(key), dict):
        overrides[key] = {**overrides[key], **value}
    else:
        overrides[key] = value
config_path = Path(CONFIG_FILE) if Path(CONFIG_FILE).is_absolute() else PROJECT / CONFIG_FILE
CONFIG = night.load_night_config(config_path, **overrides)
EXPERIMENT, NIGHT = CONFIG.experiment, CONFIG.night
INPUTS_DIR, OUTPUT_DIR, REPORTS_DIR = pipeline.resolve_roots(EXPERIMENT, PROJECT)

runtime = environment.inspect_runtime(OUTPUT_DIR)
gpu = runtime["gpu"]
print(f"Python {runtime['python']} on {runtime['platform']} (Colab: {runtime['in_colab']})")
print(
    f"GPU: {gpu['name']} {gpu['memory_total_gb']} GB, driver CUDA {gpu['driver_cuda_version']}"
    if gpu
    else "GPU: none detected (super-resolution falls back to Apple MPS or the CPU, much slower)"
)
print(f"PyTorch: {runtime['packages'].get('torch', 'not installed')}")
SR = NIGHT.super_resolution
if not SR.enabled:
    print("super-resolution: disabled (the night input is the Lanczos baseline, no SR applied)")
elif sr_models.available():
    sr_device = sr_models.device_info(sr_models.resolve_device(SR.device))
    print(f"super-resolution: {SR.backend} on {sr_device['name']} ({sr_device['device']}, torch {sr_device['torch']})")
else:
    print(f"WARNING: the SR stage will fail: {pipeline.SR_INSTALL_HINT}")
print(
    f"RAM: {runtime['ram_available_gb']} of {runtime['ram_total_gb']} GB available; "
    f"free disk: {runtime['disk_free_gb']} GB; CPUs: {runtime['cpu_count']}"
)
print(f"profile {EXPERIMENT.profile}; masters under {OUTPUT_DIR / EXPERIMENT.input_category}")

## 5. Input upload

`upload` asks for a file in Colab, `path` uses `INPUT_PATH` (for example a file on Drive), and `synthetic`
generates the deterministic synthetic night scene (`SYNTHETIC_SIZE`). Supported formats: PNG, JPEG and TIFF;
WebP, AVIF and OpenEXR when the runtime can decode them. The original file is never modified.

In [ ]:
# Uploads and generated inputs are staged outside the read-only input store, which ingest() copies into.
STAGING = OUTPUT_DIR / "_staging"

if INPUT_MODE == "upload":
    if not IN_COLAB:
        raise RuntimeError("upload only works in Colab; use INPUT_MODE='path' or 'synthetic'")
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError(f"upload exactly one image, got {len(uploaded)}")
    name, data = next(iter(uploaded.items()))
    STAGING.mkdir(parents=True, exist_ok=True)
    SOURCE = STAGING / name
    SOURCE.write_bytes(data)
elif INPUT_MODE == "path":
    SOURCE = Path(INPUT_PATH).expanduser()
elif INPUT_MODE == "synthetic":
    width, height = (int(v) for v in SYNTHETIC_SIZE.lower().split("x"))
    SOURCE = night.write_night_scene(STAGING / f"synthetic_night_{width}x{height}.png", width, height)
else:
    raise ValueError(f"unknown INPUT_MODE {INPUT_MODE!r}")
print(f"source: {SOURCE} ({SOURCE.stat().st_size:,} bytes)")
display(Image(filename=str(SOURCE), width=720))

## 6. Input validation

Copies the source into the read-only input store, decodes it without channel swapping or tone changes, and
checks dimensions, aspect ratio, bit depth, channels, alpha, ICC profile, EXIF, file size, clipping,
histograms, compression artefacts, corruption and duplicates (PRD 3.4, 7.1). Clipping at the maximum code is
expected for night scenes with practical lights; section 15 reports it before and after processing.

In [ ]:
source_image = ingestion.ingest(
    SOURCE, INPUTS_DIR, EXPERIMENT.input_category, source_color_space=EXPERIMENT.source_color_space
)
v = source_image.validation
rows = [
    ("format", f"{v['file']['format']}, {v['file']['size_bytes']:,} bytes, sha256 {v['file']['sha256'][:16]}..."),
    ("stored read-only copy", v["file"]["stored_copy"]),
    (
        "dimensions",
        f"{v['dimensions']['width']} x {v['dimensions']['height']} "
        f"({v['dimensions']['megapixels']:.2f} MP, {v['dimensions']['aspect_ratio_reduced']})",
    ),
    ("bit depth", f"{v['samples']['bit_depth']}-bit (effective {v['samples']['effective_bit_depth']}-bit)"),
    ("channels / alpha", f"{v['samples']['channels_in_file']} / {v['alpha']}"),
    ("ICC profile", v["color"]["icc_profile_description"] or "none"),
    ("interpretation", v["color"]["resolved"]["description"] + f" ({v['color']['resolved']['origin']})"),
    ("working signal", v["color"]["working_signal"]),
    ("EXIF", v["metadata"]["exif"] or "none"),
    ("clipping at max", {k: f"{c['at_max_fraction']:.3%}" for k, c in v["clipping"]["channels"].items()}),
    ("8x8 blockiness", f"{v['compression']['blockiness_ratio']:.3f}"),
    ("duplicates", v["duplicates"] or "none"),
]
display(table(["check", "result"], rows))
for warning in source_image.warnings:
    print("WARNING:", warning)

## 7. Experiment configuration

Every parameter of the profile loaded in section 4, with its accepted values, effect and warning (PRD 22.2):
first the shared experiment settings, then the Layer 5B night controls (PRD 19.1; the 19.2 user controls are
marked in `configs/night_default.yaml`). Then the exact target resolution (PRD 3.5), the memory estimate
(PRD 21.3) and the storage estimate (PRD 21.5).

In [ ]:
display(
    table(
        ["parameter", "value", "default", "accepted", "effect", "warning"],
        [
            (f"`{r['parameter']}`", f"`{r['current']}`", f"`{r['default']}`", r["accepted"], r["effect"], r["warning"])
            for r in lab_config.parameter_table(EXPERIMENT)
        ],
    )
)
display(
    table(
        ["night parameter", "value", "default", "accepted", "description", "effect", "warning"],
        [
            (
                f"`{r['parameter']}`",
                f"`{r['current']}`",
                f"`{r['default']}`",
                r["accepted"],
                r["description"],
                r["effect"],
                r["warning"] or "-",
            )
            for r in night.parameter_table(NIGHT)
        ],
    )
)

target = resolution.compute_target_resolution(
    source_image.width,
    source_image.height,
    EXPERIMENT.effective_target_megapixels,
    mode=EXPERIMENT.resolution_mode,
    pixel_count_tolerance=EXPERIMENT.pixel_count_tolerance,
    dimension_multiple=EXPERIMENT.dimension_multiple,
    preserve_aspect_ratio=EXPERIMENT.preserve_aspect_ratio,
    max_aspect_ratio_error=EXPERIMENT.max_aspect_ratio_error,
)
print(
    f"target: {target.width} x {target.height} = {target.pixels:,} pixels "
    f"(pixel error {target.pixel_count_error:+.4%}, aspect error {target.aspect_ratio_error:+.5%}, "
    f"scale {target.scale_x:.4f} x {target.scale_y:.4f})"
)
for row in environment.estimate_memory(target.width, target.height):
    print(f"  {row['representation']:<28} {row['estimated_mb']:>8.1f} MB")
storage = environment.estimate_storage(
    target.width,
    target.height,
    1,
    tiff=EXPERIMENT.export_tiff16,
    exr=EXPERIMENT.export_exr,
    exr_bytes_per_sample=2 if EXPERIMENT.exr_precision == "half" else 4,
    source_pixels=source_image.width * source_image.height,
    save_intermediates=EXPERIMENT.save_intermediates,
)
output_free_gb = environment.disk_free_gb(OUTPUT_DIR)
print(
    f"storage upper bound: {storage['upper_bound_bytes'] / 1e9:.2f} GB; free at the output folder: {output_free_gb} GB"
)
if storage["upper_bound_bytes"] > output_free_gb * 1e9:
    print("WARNING: free disk may be insufficient; point OUTPUT_ROOT at Drive or use the quick preset")

## 8. Scene analysis

Segments the PRD 8.3 luminance regions on the source. Every pixel gets exactly one label; processing uses
feathered versions of the labels, so region borders do not show. If a region is wrong (for example a lit sign
labelled as midtone), add a correction to `REGION_OVERRIDES` in section 2 and re-run from section 4: the
corrected map is shown here, recorded in the manifest, and used by the pipeline.

In [ ]:
REGIONS = night.segment_regions(source_image.linear)
for override in REGION_OVERRIDES:
    REGIONS = REGIONS.override(override["region"], override["boxes"], override.get("note", ""))
region_path = night_pipeline.region_figure(REGIONS, source_image.linear, STAGING / "regions.png")
display(Image(filename=str(region_path)))
fractions = REGIONS.fractions()
display(table(["region", "share of source pixels"], [(name, f"{fractions[name]:.2%}") for name in night.REGION_NAMES]))
for record in REGIONS.overrides:
    print(f"override: {record['region']} at {record['where']} ({record['pixels']} pixels) {record['note']}")
del source_image  # the pipeline ingests again from the read-only copy

## 9. Baseline reconstruction

Runs the Phase 3 night pipeline, which covers sections 9 to 12 in one call so that only one full-size buffer
is resident at a time: the bicubic and Lanczos baselines and the two SR slots are each built, measured
(PRD 23.5 night metrics, source consistency), cropped and released. The night input (the configured SR mode,
or Lanczos without SR) runs last and is kept, so no SR mode runs twice; Layer 5B processes it in place,
followed by the exports and figures. Every stage is logged with its
PRD 5.2 classification, duration and peak memory. If a stage fails, outputs already written are kept and the
report names the failing stage (PRD 22.3).

In [ ]:
try:
    RESULT = night_pipeline.run_night(
        SOURCE, CONFIG, PROJECT, region_overrides=REGION_OVERRIDES, notebook_version=NOTEBOOK_VERSION
    )
except pipeline.StageError as error:
    print(f"FAILED in stage {error.stage}: {error}")
    if error.manifest_path:
        print(f"partial manifest and report: {error.manifest_path.parent}")
    print(
        "Reduced-memory retry: set RUNTIME_PRESET='quick' in the section 2 parameters cell, or a smaller "
        "super_resolution tile_size in ADVANCED, then re-run section 2 and every section after it. Outputs already "
        "written are kept and out-of-memory SR tiles are already split and retried automatically."
    )
    raise
MANIFEST = RESULT.manifest
print(
    f"run {RESULT.run_id}: {MANIFEST['status']} in {MANIFEST['total_seconds']:.1f} s, "
    f"peak RSS {MANIFEST['memory']['pipeline_peak_rss_mb']:.0f} MB"
)
display(
    table(
        ["stage", "class (PRD 5.2)", "seconds", "peak RSS MB"],
        [
            (
                s["name"],
                s["classification"],
                f"{s['seconds']:.2f}",
                f"{s['rss_peak_mb']:.0f}" if s["rss_peak_mb"] else "-",
            )
            for s in MANIFEST["stages"]
        ],
    )
)

## 10. Pretrained SR

Inference only (PRD 10), through the Phase 2 backend shared with notebook 00. Each SR slot runs the network at
its native x4 on overlapping tiles, crossfades the overlaps, resamples to the exact target size with Lanczos-3
in linear light and, unless `preserve_source_structure` is off, corrects low-frequency drift against the
source (PRD 10.3 to 10.5). One network is loaded at a time and released after its slot.

- **Mode A, conservative:** the fidelity network (SwinIR classical, pixel loss). Least invented texture.
- **Mode B, balanced:** the perceptual network (SwinIR real-world GAN) with the low-frequency correction. The
  default night input (PRD 19.1).

The tables show each slot's provenance, the device, tiling, memory fallbacks and seam gates (blend error and
hard-seam height, in 8-bit code values), then the model records: weights, architecture, licence, SHA-256,
source URL, device and precision (`docs/models.md`). The heatmaps show where each mode departs from Lanczos
interpolation, which is where detail was synthesised.

In [ ]:
display(
    table(
        ["slot", "image", "method", "provenance", "class (PRD 5.2)"],
        [
            (k, s["label"], s["method"], s["provenance"], s["classification"])
            for k, s in MANIFEST["comparisons"].items()
        ],
    )
)
print(f"night input: {MANIFEST['night']['input']['slot']} ({MANIFEST['night']['input']['label']})")
SR_RECORD = MANIFEST["super_resolution"]
if not SR_RECORD.get("enabled", True):
    print("super-resolution was not enabled for this run (ENABLE_SUPER_RESOLUTION)")
elif SR_RECORD.get("skipped"):
    print(f"super-resolution skipped: {SR_RECORD['skipped']}")
else:
    display(
        table(
            ["mode", "weights", "device", "tile / overlap", "tiles", "inference s", "fallbacks", "hard seam"],
            [
                (
                    s["preset"]["label"],
                    f"`{s['weights']}`",
                    f"{s['model']['device']['name']} {s['model']['precision']}",
                    f"{s['tiling']['tile_size']} / {s['tiling']['tile_overlap']}",
                    s["tiling"]["tiles"],
                    f"{s['tiling']['inference_seconds']:.0f}",
                    len(s["tiling"]["fallbacks"]),
                    f"{s['seams']['max_score']} ({'pass' if s['seams']['passed'] else 'FAIL'})",
                )
                for s in SR_RECORD["modes"].values()
            ],
        )
    )
    display(
        table(
            ["weights", "architecture", "licence", "SHA-256", "source"],
            [
                (
                    f"`{m['weights']['name']}`",
                    m["weights"]["architecture"],
                    m["weights"]["license"],
                    f"`{m['weights']['sha256'][:16]}...`",
                    m["weights"]["url"],
                )
                for m in MANIFEST["models"]
            ],
        )
    )
    for mode in SR_RECORD["modes"]:
        heatmap = RESULT.reports_dir / "figures" / f"sr_difference_{mode}.png"
        if heatmap.exists():
            print(f"|{mode} SR - Lanczos| (synthesised detail and drift)")
            display(Image(filename=str(heatmap)))

## 11. Specialized reconstruction

The region-aware Layer 5B steps: conservative chroma-noise control, debanding, region-scaled sharpening,
optional atmospheric local contrast and halo suppression around lights, each with the provenance label its
output carries. The PRD 12.6 failure rule compares every block with its input for halos, banding, black crush
and excess artificial texture, and falls back to a more conservative local result where one appears.

In [ ]:
TONAL_OPTICAL = {"chromatic_adaptation", "shadow_tone", "highlight_rolloff", "night_color", "dither", "bloom", "grain"}
RECORD = MANIFEST["night"]


def step_table(steps):
    return table(
        ["step", "enabled", "provenance", "class", "note"],
        [
            (s["name"], "yes" if s["enabled"] else "no", s["provenance"], s["classification"], s["note"] or "-")
            for s in steps
        ],
    )


display(step_table([s for s in RECORD["stages"] if s["name"] not in TONAL_OPTICAL]))
fallbacks = RECORD["fallbacks"]
if not any(s["name"] == "failure_rule" and s["enabled"] for s in RECORD["stages"]):
    print("failure rule: off, no blocks were checked")
else:
    print(f"failure rule: {len(fallbacks)} fallback blocks ({RECORD['fallback_fraction']:.3%} of output pixels)")
if fallbacks:
    display(
        table(
            ["x", "y", "width", "height", "reasons", "regions"],
            [
                (f["x"], f["y"], f["width"], f["height"], ", ".join(f["reasons"]), ", ".join(f["regions"]))
                for f in fallbacks[:20]
            ],
        )
    )
display(Image(filename=MANIFEST["figures"]["fallbacks"]))

## 12. Tonal and optical processing

The pointwise Layer 5B steps: chromatic adaptation, shadow toe lift with black kept at black, the moderate
highlight shoulder (which lowers clipping without recovering radiance), night colour (warm/cool balance,
saturation vs luminance, shadow neutrality, hue-preserving gamut mapping), and the simulated dither, bloom and
grain. The information record maps every step onto the manifest's preserved, interpolated, inferred and
simulated categories.

In [ ]:
display(step_table([s for s in RECORD["stages"] if s["name"] in TONAL_OPTICAL]))
display(table(["information", "steps"], list(RECORD["information"].items())))

## 13. Optional sensor simulation

Not part of Phase 3 (PRD Phase 5). `noise.synthetic_sensor_noise` stays off.

## 14. Professional exports

16-bit sRGB TIFF (embedded sRGB ICC profile, lossless Deflate, provenance in ImageDescription) and OpenEXR
(R, G, B half by default, linear Rec.709, chromaticities and scene-referred caveat in the header) under
`outputs/night/<run id>/`. Each file was re-read and checked after writing (PRD 17.1, 23.6), and each has a
JSON sidecar with the provenance labels of every step. When the night input is an SR slot, the masters carry
the notice that their fine detail was synthesised and the model weights' checksum and licence.

In [ ]:
exports = list(MANIFEST.get("intermediates", [])) + list(MANIFEST["final"]["exports"].values())
display(
    table(
        ["file", "kind", "size", "checks"],
        [
            (
                f"`{Path(e['path']).name}`",
                e["kind"],
                f"{e['size_bytes'] / 1e6:.1f} MB",
                f"{sum(e['checks'].values())}/{len(e['checks'])}",
            )
            for e in exports
        ],
    )
)
print(f"masters: {RESULT.masters_dir}")

## 15. Quality metrics

PRD 23.5 night metrics for the source, every comparison image, and the night input before and the final
after Layer 5B: highlight clipping, gradient banding, halos around bright edges, shadow chroma noise and
black crush. Then the paired comparisons (colour difference in protected light regions, shadow chroma shift,
local contrast change per region) and source consistency (PRD 23.2). A lower clipping fraction comes from the
highlight shoulder, not from recovered radiance.

In [ ]:
header, rows = night_pipeline.metrics_table(MANIFEST)
display(table(header, rows))

paired = RECORD["comparison"]
display(table(*night_pipeline.paired_table(RECORD)))
before, after = RECORD["metrics_before"]["local_contrast_codes"], RECORD["metrics_after"]["local_contrast_codes"]
display(
    table(
        ["region", "local contrast before", "after", "change ratio"],
        [
            (
                n,
                f"{before[n]:.3f}" if before[n] is not None else "-",
                f"{after[n]:.3f}" if after[n] is not None else "-",
                f"{paired['local_contrast_change'][n]:.3f}" if paired["local_contrast_change"][n] is not None else "-",
            )
            for n in before
        ],
    )
)
consistency = {k: s["source_consistency"] for k, s in MANIFEST["comparisons"].items()}
consistency[night_pipeline.FINAL] = MANIFEST["final"]["source_consistency"]
display(
    table(
        ["source consistency", *consistency],
        [
            ("PSNR (dB)", *(f"{m['psnr_db']:.2f}" for m in consistency.values())),
            ("SSIM", *(f"{m['ssim']:.5f}" for m in consistency.values())),
            ("delta E 2000 mean", *(f"{m['color_difference_ciede2000']['mean']:.4f}" for m in consistency.values())),
        ],
    )
)

## 16. Visual comparison

The PRD 19.3 comparisons on identical crops at 100% output pixels: original highlight, shadow, reflection and
gradient crops (nearest-neighbour), the bicubic and Lanczos baselines, the conservative and balanced SR slots
(Phase 2 networks, inferred detail), the final night processing and the amplified difference between night input and final.
The shadow panel adds a second row at +3 EV display gain so near-black structure can be judged; that gain is
display only and never reaches the masters or the metrics.
Then the gradient and banding diagnostics, the light-source halo comparison and the delta E map. Review them
at 100% and 200% for halos, banding, crushed blacks and invented texture (PRD 25.2, 26.2).

In [ ]:
for panel in MANIFEST["crop_panels"]:
    print(f"crop {panel['crop']} (output box {panel['box']})")
    display(Image(filename=panel["path"]))
for key, caption in (("banding", "gradient and banding diagnostics"), ("halo", "light-source halo comparison")):
    if key in MANIFEST["figures"]:
        print(caption)
        display(Image(filename=MANIFEST["figures"][key]))
display(Image(filename=str(RESULT.reports_dir / "figures" / "delta_e_night.png")))

## 17. Report generation

The Markdown report (PRD 27, with the region-mask figure, fallback map, comparisons and the before/after
metrics table) and the machine-readable manifest are written next to their figures.

In [ ]:
print(f"report:   {RESULT.report_path}")
print(f"manifest: {RESULT.manifest_path}")
display(Markdown("\n".join(f"- {line}" for line in night_pipeline.night_conclusions(MANIFEST))))

## 18. Download and archive

**The final image.** This notebook makes one master, the Layer 5B night processing of the night input (section 11). It is saved again under `final/` in the masters
folder with `final_` in front of its name, verified byte for byte, and shown below.

**Delivery.** In Colab, `DOWNLOAD_FINAL` downloads the final 16-bit TIFF: **about 235 MB** for a standard 50 MP
run, so the browser download takes a few minutes (if it stalls, take the file from Drive or the Files panel).
With Drive mounted (`MOUNT_DRIVE`), the final TIFF, EXR and their provenance sidecars (about 400 MB) are copied
to `FINAL_DRIVE_DIR/<run id>`, unless the masters are on Drive already.

**Report archive.** The report and figures are zipped and downloaded separately (PRD 21.5); the zip holds no
masters.

In [ ]:
from IPython.display import Image, Markdown, display

from synthetic_raw_lab import final_output

FINAL = MANIFEST.get("final_output") or {}
if FINAL.get("files"):
    display(Markdown(f"**Final 50 MP image:** {FINAL['reason']}."))
    preview = RESULT.reports_dir / "figures" / "final_preview.jpg"
    if preview.exists():
        display(Image(filename=str(preview)))
    for f in FINAL["files"].values():
        print(f"{Path(f['path']).name}: {f['width']} x {f['height']}, {f['size_bytes'] / 1e6:,.0f} MB ({f['path']})")
DELIVERY = final_output.deliver(MANIFEST, download=DOWNLOAD_FINAL, drive_dir=FINAL_DRIVE_DIR if MOUNT_DRIVE else None)

In [ ]:
import shutil

archive = shutil.make_archive(str(RESULT.reports_dir), "zip", root_dir=RESULT.reports_dir)
print(f"report archive (report and figures, no masters): {archive}")
if IN_COLAB:
    from google.colab import files

    files.download(archive)